# Семинар 06. ООП и принципы SOLID


## Цели

После семинара вы сможете:

- распознавать нарушения принципов SOLID в небольших примерах;
- разделять ответственности между классами;
- проектировать расширяемые зависимости через абстракции.

## Перед началом

Нужны классы, наследование, композиция и абстрактные базовые классы.


## Что такое SOLID

SOLID — пять принципов проектирования модулей и зависимостей:

- **S** — Single Responsibility Principle, принцип единственной ответственности;
- **O** — Open/Closed Principle, принцип открытости/закрытости;
- **L** — Liskov Substitution Principle, принцип подстановки Лисков;
- **I** — Interface Segregation Principle, принцип разделения интерфейса;
- **D** — Dependency Inversion Principle, принцип инверсии зависимостей.

Они нужны, чтобы изменение одного правила не заставляло переписывать половину проекта и затем искать случайные поломки. Если модуль знает обо всём, делает всё и напрямую создаёт все свои зависимости, это не простота, а стоимость, отложенная до следующего изменения.

SOLID — не закон природы и не повод заранее строить двадцать интерфейсов вокруг трёх функций. Принципы применяют там, где уже видна ось изменений. Бессмысленная абстракция портит код не хуже, чем отсутствие нужной абстракции.

## S: Single Responsibility Principle

У модуля должна быть одна причина для изменения — один источник требований, или актор. Актор здесь не обязательно один человек: это может быть бизнес-подразделение, команда API или команда инфраструктуры.

Фраза «класс работает с пользователем» ещё не описывает одну ответственность. Правила расчёта скидки меняет бизнес, формат JSON — контракт API, а сохранение в PostgreSQL — инфраструктура. Если всё это находится в `UserService`, класс меняется по трём независимым причинам и связывает решения, которые должны меняться отдельно.

Простой тест: если в описании класса естественно появляется несколько независимых союзов «и», класс, скорее всего, делает лишнее. Делить код на классы по одному методу тоже не надо: цель SRP — высокая связность внутри модуля и слабая связанность между разными причинами изменения.

## O: Open/Closed Principle

Стабильный модуль должен позволять добавить ожидаемый вариант поведения без переписывания уже проверенной основной логики. Например, расчёт суммы на кассе не должен меняться при подключении нового способа оплаты: новый обработчик реализует существующий платёжный контракт, а касса продолжает работать с этим контрактом.

Расширение может происходить через композицию, функцию-стратегию, конфигурацию, регистрацию обработчика или наследование. Создавать подкласс на каждое изменение принцип не требует.

OCP работает только относительно конкретной оси изменений. Невозможно закрыть модуль от всех будущих изменений, потому что будущее не обязано уважать наши догадки. Сначала находят реально меняющуюся часть, затем ставят границу абстракции именно там.

## L: Liskov Substitution Principle

Объект подтипа должен заменять объект базового типа, не ломая обоснованные ожидания клиентского кода. Подтип не должен усиливать предусловия, ослаблять постусловия или нарушать инварианты базового типа.

Сам факт `class A(B)` ничего не гарантирует. Он только сообщает интерпретатору о наследовании. Если `A` меняет смысл операций `B`, это плохой подтип, сколько бы методов он ни унаследовал. Классический пример — изменяемые прямоугольник и квадрат:


In [ ]:
class Rectangle:
    def __init__(self, width: int, height: int) -> None:
        self.width = width
        self.height = height
    
    def area(self) -> int:
        return self.width * self.height

    def set_width(self, width: int) -> None:
        self.width = width

    def set_height(self, height: int) -> None:
        self.height = height


class Square(Rectangle):
    def __init__(self, side: int) -> None:
        super().__init__(side, side)
    
    def set_width(self, width: int) -> None:
        self.width = width
        self.height = width

    def set_height(self, height: int) -> None:
        self.width = height
        self.height = height


def resize_to_5_by_4(rectangle: Rectangle) -> None:
    rectangle.set_width(5)
    rectangle.set_height(4)
    assert rectangle.area() == 20


resize_to_5_by_4(Rectangle(3, 4))
# Square нарушает ожидание независимого изменения сторон: assertion упадёт.
# resize_to_5_by_4(Square(3))


Квадрат математически является прямоугольником, но изменяемый `Square` не является поведенческим подтипом данного изменяемого `Rectangle`. Клиент вправе ожидать, что `set_height()` не меняет ширину. Нормальное решение — не наследовать эти классы друг от друга: сделать их независимыми типами либо дать обоим общий неизменяемый интерфейс фигуры с методом `area()`.

[Исходная работа Барбары Лисков о подстановке и иерархиях типов](https://www.cs.tufts.edu/~nr/cs257/archive/barbara-liskov/data-abstraction-and-hierarchy.pdf).

## I: Interface Segregation Principle

Клиент не должен зависеть от методов, которые ему не нужны. Толстый интерфейс `Animal` с методами `fly()`, `walk()` и `swim()` требует от каждого животного реализовать даже невозможные для него операции. Заглушка `raise NotImplementedError` не исправляет дизайн: она показывает, что объект на самом деле не выполняет заявленный контракт.

Интерфейс делят по потребностям клиентов. Один класс может реализовать несколько узких интерфейсов:

```python
from abc import ABC, abstractmethod


class Flyable(ABC):
    @abstractmethod
    def fly(self) -> None:
        ...


class Bird(Flyable):
    def fly(self) -> None:
        print("I'm flying")


class Airplane(Flyable):
    def fly(self) -> None:
        print("I'm flying with engines")


class Walkable(ABC):
    @abstractmethod
    def walk(self) -> None:
        ...


class Dog(Walkable):
    def walk(self) -> None:
        print("I'm walking")


class Duck(Flyable, Walkable):
    def fly(self) -> None:
        print("I'm flying")

    def walk(self) -> None:
        print("I'm walking")
```

## D: Dependency Inversion Principle

- Модуль с бизнес-правилом не должен зависеть от конкретной базы, HTTP-клиента или устройства. И политика, и детали зависят от абстракции.
- Абстракцию формулируют в терминах потребности высокоуровневого кода. Например, сервису заказов нужна операция `save_order(order)`, а не методы конкретной SQL-библиотеки. Репозиторий реализует нужный сервису контракт; инфраструктура не диктует бизнес-логике свой API.

Если `FlyablesManager` сам создаёт `Bird()` и `Airplane()`, он напрямую зависит от этих классов и сам решает, какие реализации допустимы. Передача зависимостей снаружи убирает эту связь:

```python
from collections.abc import Iterable


class FlyablesManager:
    def __init__(self, flyables: Iterable[Flyable] = ()) -> None:
        self._flyables = list(flyables)

    def add_flyable(self, flyable: Flyable) -> None:
        self._flyables.append(flyable)

    def fly_all(self) -> None:
        for flyable in self._flyables:
            flyable.fly()


manager = FlyablesManager([Bird(), Airplane()])
manager.fly_all()
```

Передача зависимости через конструктор — dependency injection. Это техника, а не сам DIP: можно внедрить десять конкретных инфраструктурных классов и всё равно сохранить плохое направление зависимостей. Суть DIP — кто определяет контракт и от чего зависит важная логика.


## Как принципы связаны

| Проблема в коде | Что бьёт тревогу | Что обычно исправляют |
|---|---|---|
| Один класс меняется из-за бизнес-правил, формата API и базы | SRP | разделяют независимые ответственности |
| Для каждого нового варианта приходится лезть в старую цепочку `if/elif` | OCP | выделяют реальную ось изменений и точку расширения |
| Подкласс требует особых проверок или ломает ожидания базы | LSP | исправляют контракт, композицию или иерархию |
| Реализации содержат заглушки для ненужных методов | ISP | делят толстый интерфейс по потребностям клиентов |
| Бизнес-логика напрямую создаёт инфраструктурные детали | DIP | разворачивают зависимость через контракт и внедрение реализации |

Принципы пересекаются: одна неудачная граница часто нарушает сразу несколько из них. Важно не подобрать букву из аббревиатуры, а сделать следующее изменение локальным и предсказуемым. Если после «улучшения по SOLID» код стал длиннее, а изменение по-прежнему приходится вносить по всему проекту, улучшения не произошло.


## Самопроверка

1. Почему «один метод на класс» не является правильным толкованием SRP?
2. От каких изменений модуль вообще может быть закрыт согласно OCP?
3. Почему наследование `Square(Rectangle)` не доказывает выполнение LSP?
4. Что не так с реализацией интерфейса через `raise NotImplementedError` для половины методов?
5. Чем DIP отличается от передачи любого объекта через конструктор?
6. В какой момент новая абстракция помогает, а в какой только увеличивает объём кода?


## Итоги

- SRP отделяет причины изменения, а не считает методы.
- OCP защищает стабильную логику от ожидаемого варианта изменений, но не от любого будущего.
- LSP проверяет поведение подтипа с точки зрения клиента, а не форму иерархии.
- ISP не заставляет клиентов и реализации зависеть от ненужных операций.
- DIP направляет зависимость от деталей к контракту, нужному бизнес-логике.
- SOLID полезен как набор диагностических принципов; механическое следование аббревиатуре производит архитектурный мусор.


## Задание 1. Blackjack (до 2 баллов)

Реализуйте упрощённую игру в Blackjack. Начните со следующих классов:

```python
import abc


class Card:
    def __init__(self, number):
        # TODO: проверить, что number находится в диапазоне от 0 до 51.
        self._number = number
    
    def __str__(self):
        # Верните номинал и масть, например 10❤️.
        # Порядок: пики, трефы, бубны, черви; внутри масти — от 2 до A.
        pass


class CardValueManager(abc.ABC):
    @abc.abstractmethod
    def get_value(self, card: Card) -> int:
        """
        Вернуть возможные значения карты.

        Карты от 2 до 10 стоят по номиналу, J, Q и K — по 10 очков,
        туз — 1 или 11. При необходимости измените возвращаемый тип метода,
        чтобы поддержать оба значения туза.
        """
        pass

class Deck:
    # Перемешайте колоду и сохраните её состояние в экземпляре.
    
    def next_card(self) -> Card:
        # Верните следующую карту. Для последовательной выдачи используйте генератор.
        # Если колода закончилась, явно сообщите об этом вызывающему коду.
        pass

class Player:
    # Храните карты в руке, сумму очков и статус: игра продолжается, победа или поражение.
    # Игрок должен уметь получить карту из колоды.
    pass

class GameEngine:
    """
    Управляет одной партией.

    В игре участвуют игрок и дилер. В начале игрок получает две открытые карты,
    дилер — одну открытую карту. Затем ходит игрок.

    Игрок может брать карты или остановиться. После каждой новой карты сумма
    пересчитывается: больше 21 — поражение, ровно 21 — победа. Если игрок
    остановился раньше, ход переходит к дилеру.

    Дилер берёт карты, пока сумма меньше 17. После каждой карты проверяются
    перебор и ровно 21. Если партия не завершилась, сравните суммы игрока
    и дилера; предусмотрите победу каждой стороны и ничью.
    """
    pass

```

Каждый класс поместите в отдельный файл.



**Оценивание и критерии проверки**

- 1 балл: реализованы `Card`, `CardValueManager` и `Deck`, проверяются границы номера карты и окончание колоды;
- 2 балла: дополнительно реализованы `Player` и `GameEngine`, включая туз как 1 или 11, перебор дилера до 17, победу, поражение и ничью;
- каждый класс находится в отдельном модуле, а правила игры покрыты тестами без пользовательского ввода внутри бизнес-логики.
